# Root Locus

For a unity-feedback system with open-loop TF $G(s) = K \dfrac{N(s)}{D(s)}$ ($m$ zeros, $n \geq m$ poles), the **closed-loop poles** are the roots of:

$$D(s) + K\,N(s) = 0 \tag{Characteristic Equation}$$

A point $s$ belongs to the **Positive Root Locus** ($K \geq 0$) if and only if:

$$\angle G(s) = (2q+1)\cdot 180^\circ, \quad q \in \mathbb{Z}$$

The **gain** at any $s^*$ on the locus is recovered from $K = |D(s^*)|\,/\,|N(s^*)| = \prod_i|s^*-p_i|\,/\,\prod_j|s^*-z_j|$.

---

### Five Construction Rules

**1. Branches & symmetry** — $n$ branches, symmetric about the real axis.
Branches *start* at open-loop poles ($K=0$) and *end* at zeros ($K\to\infty$); $n-m$ branches diverge to $\infty$.

**2. Real-axis rule** — A real point belongs to the RL if the count of real poles + zeros **to its right** is **odd**.

**3. Asymptotes** — $n-m$ diverging branches follow rays from centroid $\sigma_a$ at angles $\phi_k$:
$$\sigma_a = \frac{\displaystyle\sum_i \mathrm{Re}(p_i) - \sum_j \mathrm{Re}(z_j)}{n-m}, \qquad \phi_k = \frac{(2k+1)\cdot 180^\circ}{n-m}, \quad k = 0,\ldots, n-m-1$$

**4. Break points** — Branches meet / split on the real axis at real roots of:
$$N'(s)\,D(s) - N(s)\,D'(s) = 0$$

**5. Departure angle** — The positive RL leaves complex pole $p_i$ at angle:
$$\phi_{\!\mathrm{dep}} = 180^\circ - \sum_{j \neq i}\angle(p_i - p_j) + \sum_{j}\angle(p_i - z_j)$$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive, FloatSlider, Checkbox
from IPython.display import display
%matplotlib inline
plt.rcParams['figure.dpi'] = 100


def _poly(roots):
    return np.poly(np.array(roots, dtype=complex))


def rl_compute(zeros, poles, k_max=200, n_pts=600):
    # Positive-RL branches for K in [0, k_max], sorted for branch continuity
    n = len(poles)
    num, den = _poly(zeros), _poly(poles)
    k_vals = np.linspace(0, k_max, n_pts)
    B = np.zeros((n_pts, n), dtype=complex)
    for i, k in enumerate(k_vals):
        B[i] = np.roots(np.polyadd(den, k * num))
    for i in range(1, n_pts):
        prev, curr = B[i - 1], B[i].copy()
        avail = list(range(n))
        for j in range(n):
            idx = avail[np.argmin([abs(curr[a] - prev[j]) for a in avail])]
            B[i, j] = curr[idx]
            avail.remove(idx)
    return k_vals, B


def get_asymptotes(zeros, poles):
    n, m = len(poles), len(zeros)
    if n <= m:
        return None, []
    c = (sum(p.real for p in poles) - sum(z.real for z in zeros)) / (n - m)
    return c, [(2 * k + 1) * 180 / (n - m) for k in range(n - m)]


def get_breakpoints(zeros, poles):
    num, den = _poly(zeros), _poly(poles)
    dn = np.polyder(num) if len(num) > 1 else np.zeros(1)
    dd = np.polyder(den) if len(den) > 1 else np.zeros(1)
    bp = np.polysub(np.polymul(dn, den), np.polymul(num, dd))
    return np.roots(bp) if len(bp) > 1 else np.array([])


def get_departure_angles(zeros, poles):
    res = {}
    for i, p in enumerate(poles):
        if abs(p.imag) > 1e-6:
            sp = sum(np.angle(p - poles[j]) for j in range(len(poles)) if j != i)
            sz = sum(np.angle(p - z) for z in zeros)
            res[i] = np.degrees(np.pi - sp + sz) % 360
    return res


def _ax_setup(ax, pa, za, lim):
    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel('Re(s)')
    ax.set_ylabel('Im(s)')
    ax.grid(alpha=0.3)
    ax.plot(pa.real, pa.imag, 'rx', ms=12, mew=2.5, zorder=5, label='Poles (K=0)')
    if len(za):
        ax.plot(za.real, za.imag, 'bo', ms=10, mew=2, mfc='none', zorder=5, label='Zeros (K->inf)')


# Basic Root Locus Explorer: move poles and zeros, observe how branches change

def plot_explorer(p1, p2, p3, z1, z2, k_max):
    poles = [p1 + 0j, p2 + 0j, p3 + 0j]
    zeros = [z1 + 0j, z2 + 0j]
    k_vals, B = rl_compute(zeros, poles, k_max=k_max)
    pa, za = np.array(poles), np.array(zeros)

    fig, ax = plt.subplots(figsize=(8, 6))
    for j, col in enumerate(['steelblue', 'darkorange', 'seagreen']):
        ax.plot(B[:, j].real, B[:, j].imag, color=col, lw=1.8, label=f'Branch {j + 1}')
    lim = max(6.0, float(abs(pa).max()), float(abs(za).max())) + 2.0
    _ax_setup(ax, pa, za, lim)
    ax.set_title(f'Root Locus   n=3, m=2   K in [0, {k_max:.0f}]')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


sw = dict(style={'description_width': '60px'}, layout=widgets.Layout(width='275px'))
display(interactive(plot_explorer,
    p1=FloatSlider(-1.0, min=-7, max=3, step=0.2, description='Pole 1', **sw),
    p2=FloatSlider(-3.0, min=-7, max=3, step=0.2, description='Pole 2', **sw),
    p3=FloatSlider(-5.0, min=-7, max=3, step=0.2, description='Pole 3', **sw),
    z1=FloatSlider(-0.5, min=-7, max=3, step=0.2, description='Zero 1', **sw),
    z2=FloatSlider(-2.0, min=-7, max=3, step=0.2, description='Zero 2', **sw),
    k_max=FloatSlider(50.0, min=5, max=300, step=5, description='K max', **sw),
))

interactive(children=(FloatSlider(value=-1.0, description='Pole 1', layout=Layout(width='275px'), max=3.0, min…

## Geometric Aids & Stability

**Stability** — a closed-loop system is stable when **all** poles have $\mathrm{Re}(s) < 0$. The imaginary axis is the stability boundary.

**Critical gain** $K_{\mathrm{crit}}$ — the value of $K$ at which a branch crosses the $j\omega$-axis. For branches starting in the RHP ($\mathrm{Re} > 0$), the system is unstable until $K > K_{\mathrm{crit}}$ pushes them into the LHP.

The interactive below adds three geometric aids on top of the root locus:

- **Asymptote rays** from the centroid $\sigma_a$ (gray dashed lines + square marker)
- **Break-away / break-in** points on the real axis (orange stars)
- **Departure angle** arrows from complex poles (purple), computed with Rule 5

Try moving the complex poles into the RHP ($\mathrm{Re} > 0$) and observe how the centroid and departure angles change.

In [ ]:
def plot_geometric(re_cp, im_cp, re_rp, re_z, show_asym, show_bp, show_dep, show_shade):
    poles = [complex(re_cp, im_cp), complex(re_cp, -im_cp), re_rp + 0j]
    zeros = [re_z + 0j]
    pa, za = np.array(poles), np.array(zeros)
    k_vals, B = rl_compute(zeros, poles, k_max=150)
    c, angs = get_asymptotes(zeros, poles)

    fig, ax = plt.subplots(figsize=(9, 7))

    title_extra = ''
    if show_shade:
        ax.axvspan(-20, 0, alpha=0.06, color='green')
        ax.axvspan(0,  20, alpha=0.06, color='red')
        crossings = []
        for j in range(len(poles)):
            rv = B[:, j].real
            for ii in range(len(rv) - 1):
                if rv[ii] * rv[ii + 1] < 0:
                    kc = np.interp(0, [rv[ii], rv[ii + 1]], [k_vals[ii], k_vals[ii + 1]])
                    crossings.append(kc)
        if crossings:
            title_extra = f'   K_crit = {min(crossings):.2f}'
        else:
            title_extra = '   (no jw crossing in range)'

    for j in range(len(poles)):
        ax.plot(B[:, j].real, B[:, j].imag, lw=1.8)

    ax.plot(pa.real, pa.imag, 'rx', ms=12, mew=2.5, zorder=5)
    ax.plot(za.real, za.imag, 'bo', ms=10, mew=2, mfc='none', zorder=5)

    if show_asym and c is not None:
        ax.plot(c, 0, 's', ms=9, color='gray', zorder=6, label=f'Centroid  sa={c:.2f}')
        for ang in angs:
            rad = np.deg2rad(ang)
            ax.plot([c, c + 40 * np.cos(rad)], [0, 40 * np.sin(rad)],
                    '--', color='gray', lw=1.5, alpha=0.8)

    if show_bp:
        bp = get_breakpoints(zeros, poles)
        real_bp = [r for r in bp if abs(r.imag) < 0.05 and abs(r.real) < 12]
        if real_bp:
            ax.plot([r.real for r in real_bp], np.zeros(len(real_bp)),
                    '*', ms=14, color='orange', zorder=6, label='Break points')

    if show_dep:
        dep = get_departure_angles(zeros, poles)
        for i, ang in dep.items():
            p = poles[i]
            rad = np.deg2rad(ang)
            r = 0.7
            ax.annotate('', xy=(p.real + r * np.cos(rad), p.imag + r * np.sin(rad)),
                        xytext=(p.real, p.imag),
                        arrowprops=dict(arrowstyle='->', color='purple', lw=2.0))
            ax.text(p.real + 1.15 * np.cos(rad), p.imag + 1.15 * np.sin(rad),
                    f'{ang:.0f}', color='purple', fontsize=9, ha='center', va='center')

    _ax_setup(ax, pa, za, lim=8)
    ax.set_title('Root Locus with Geometric Aids' + title_extra)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


sw2 = dict(style={'description_width': '150px'}, layout=widgets.Layout(width='390px'))
display(interactive(plot_geometric,
    re_cp    =FloatSlider(0.5,  min=-4,  max=3,   step=0.1, description='Re(complex poles)', **sw2),
    im_cp    =FloatSlider(2.0,  min=0.1, max=5,   step=0.1, description='Im(complex poles)', **sw2),
    re_rp    =FloatSlider(-3.0, min=-7,  max=2,   step=0.1, description='Real pole',         **sw2),
    re_z     =FloatSlider(-1.0, min=-6,  max=2,   step=0.1, description='Real zero',         **sw2),
    show_asym =Checkbox(True,  description='Asymptotes'),
    show_bp   =Checkbox(True,  description='Break points'),
    show_dep  =Checkbox(True,  description='Departure angles'),
    show_shade=Checkbox(True,  description='Stable/Unstable shading'),
))

interactive(children=(FloatSlider(value=0.5, description='Re(complex poles)', layout=Layout(width='390px'), ma…

## Controller Design via Root Locus

The goal is to choose a controller $C(s)$ so that the open-loop root locus has branches in the left half-plane for some achievable gain.

---

**Case $n - m = 1$** — one asymptote pointing along the negative real axis ($180^\circ$).
A pure proportional gain $C(s) = K_c$ is sufficient: increasing $K_c$ pushes all branches toward $-\infty$.

**Case $n - m = 2$** — two asymptotes at $\pm 90^\circ$ from centroid $\sigma_a$.
If $\sigma_a > 0$, branches escape into the RHP for large $K$ — a pure gain cannot stabilize.

Fix: add a **lead compensator** $C(s) = K_c \dfrac{s + z_b}{s + p_b}$ with $p_b > z_b > 0$.
This adds one zero at $-z_b$ and one pole at $-p_b$ to the open-loop system, shifting the centroid:
$$\sigma_a^{\mathrm{new}} = \frac{\sum \mathrm{Re}(p_i^{\mathrm{plant}}) - p_b - \bigl(\sum \mathrm{Re}(z_j^{\mathrm{plant}}) - z_b\bigr)}{n_{\mathrm{new}} - m_{\mathrm{new}}}$$
Choose $p_b - z_b$ large enough so that $\sigma_a^{\mathrm{new}} < 0$.

**Case $n - m \geq 3$** — reduce to $n - m = 2$ by adding zeros.
Each zero $-z_k$ must be paired with a far-off real pole $-p_k$ ($p_k \gg z_k$) to keep the controller proper:
$C(s) = K_c \prod_k \dfrac{s + z_k}{s + p_k}$.
The far-off poles barely affect the low-frequency behaviour but make $C(s)$ physically realizable.

In [ ]:
nm_drop = widgets.Dropdown(
    options=[('n - m = 1   (pure gain)', 1), ('n - m = 2   (lead compensator)', 2)],
    description='Case:',
    style={'description_width': '50px'},
    layout=widgets.Layout(width='290px'))

Kc_s = FloatSlider(2.0, min=0.1, max=30, step=0.1, description='Kc',
                   style={'description_width': '30px'}, layout=widgets.Layout(width='240px'))
zb_s = FloatSlider(0.5, min=0.1, max=8,  step=0.1, description='z_b',
                   style={'description_width': '30px'}, layout=widgets.Layout(width='240px'))
pb_s = FloatSlider(4.0, min=0.5, max=20, step=0.2, description='p_b',
                   style={'description_width': '30px'}, layout=widgets.Layout(width='240px'))

out6 = widgets.Output()


def _update_design(_):
    with out6:
        out6.clear_output(wait=True)
        Kc, zb, pb = Kc_s.value, zb_s.value, pb_s.value
        case = nm_drop.value

        if case == 1:
            ol_zeros, ol_poles = [], [1.0]
            info = 'P(s) = 1/(s-1)     C(s) = Kc     [n-m=1, single asymptote at 180 deg]'
            zb_s.layout.display = 'none'
            pb_s.layout.display = 'none'
        else:
            ol_zeros = [-zb]
            ol_poles = [1.0, 1.0, -pb]
            sa = (sum(p for p in ol_poles) - sum(z for z in ol_zeros)) / (len(ol_poles) - len(ol_zeros))
            info = (f'P(s)=1/(s-1)^2   C(s)=Kc*(s+{zb:.1f})/(s+{pb:.1f})'
                    f'   sigma_a={sa:.2f}  ({"stable dir." if sa < 0 else "UNSTABLE dir., increase pb"})')
            zb_s.layout.display = ''
            pb_s.layout.display = ''

        k_vals, B = rl_compute(ol_zeros, ol_poles, k_max=40, n_pts=600)
        num = _poly(ol_zeros)
        den = _poly(ol_poles)
        cl_poles = np.roots(np.polyadd(den, Kc * num))
        pa = np.array([complex(p) for p in ol_poles])
        za = np.array([complex(z) for z in ol_zeros]) if ol_zeros else np.array([])

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

        for j in range(len(ol_poles)):
            ax1.plot(B[:, j].real, B[:, j].imag, lw=1.8)
        ax1.plot(pa.real, pa.imag, 'rx', ms=12, mew=2.5)
        if len(za):
            ax1.plot(za.real, za.imag, 'bo', ms=10, mew=2, mfc='none')
        ax1.plot(cl_poles.real, cl_poles.imag, 'k^', ms=10, zorder=6,
                 label=f'CL poles  Kc={Kc:.1f}')
        ax1.axvline(0, color='green', lw=1.2, ls='--', alpha=0.6)
        _ax_setup(ax1, pa, za, lim=8)
        ax1.set_title('Open-Loop Root Locus')
        ax1.legend(fontsize=9)

        stable = all(p.real < 0 for p in cl_poles)
        clr = 'green' if stable else 'red'
        ax2.axvspan(-12, 0, alpha=0.07, color='green')
        ax2.axvspan(0,  12, alpha=0.07, color='red')
        ax2.plot(cl_poles.real, cl_poles.imag, '^', ms=13, color=clr, zorder=6)
        for cp in cl_poles:
            ax2.annotate(f'  {cp.real:.2f}{cp.imag:+.2f}j', (cp.real, cp.imag), fontsize=9)
        status = 'STABLE' if stable else 'UNSTABLE'
        tick = chr(10003) if stable else chr(10007)
        ax2.set_title(f'Closed-Loop Poles   {tick} {status}', color=clr, fontweight='bold')
        ax2.axhline(0, color='k', lw=0.5)
        ax2.axvline(0, color='k', lw=0.5)
        ax2.set_xlim(-8, 4)
        ax2.set_ylim(-6, 6)
        ax2.set_xlabel('Re(s)')
        ax2.set_ylabel('Im(s)')
        ax2.grid(alpha=0.3)

        fig.suptitle(info, fontsize=10)
        plt.tight_layout()
        plt.show()


for _w in [nm_drop, Kc_s, zb_s, pb_s]:
    _w.observe(_update_design, 'value')

_update_design(None)
display(widgets.VBox([widgets.HBox([nm_drop, Kc_s, zb_s, pb_s]), out6]))